<h1>Podcast Prediction time</h1>

<h2>LightGBM Regression Pipeline Notebook</h2>

In [1]:
!pip install optuna-integration[lightgbm]


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 3.4 MB/s eta 0:00:00


In [2]:
# 1. Import libraries
import pandas as pd
import numpy as np
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
import optuna
from optuna.integration import LightGBMPruningCallback

In [3]:
warnings.filterwarnings("ignore", category=RuntimeWarning)


In [4]:
# 2. Set up data paths
data_dir = "/kaggle/working"
input_dir = "/kaggle/input/playground-series-s5e4"
splits_dir = os.path.join(data_dir, "splits")
visuals_dir = os.path.join(data_dir, "visuals")
os.makedirs(splits_dir, exist_ok=True)
os.makedirs(visuals_dir, exist_ok=True)

In [5]:

train_path = os.path.join(input_dir, "train.csv")
test_path = os.path.join(input_dir, "test.csv")
sample_submission_path = os.path.join(input_dir, "sample_submission.csv")

In [6]:
# 3. Load the datasets
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_submission_path)

In [7]:
# 4. Define target and features
target_column = 'Listening_Time_minutes'
X = train.drop(columns=[target_column])
y = train[target_column]

In [8]:
# 5. Log transform target
y = np.log1p(y)

In [9]:
# 6. Combine train and test for preprocessing consistency
all_data = pd.concat([X, test], axis=0)

In [10]:
# 7. Encode categorical features using LabelEncoder
for col in all_data.select_dtypes(include='object').columns:
    le = LabelEncoder()
    all_data[col] = le.fit_transform(all_data[col].astype(str))

In [14]:
# 7b. Robust Feature Engineering based on available columns
# Interaction Features
all_data['Host_Guest_Interaction'] = all_data['Host_Popularity_percentage'] * all_data['Guest_Popularity_percentage']
all_data['Length_Ads_Interaction'] = all_data['Episode_Length_minutes'] * all_data['Number_of_Ads']
all_data['Time_Sentiment_Interaction'] = all_data['Publication_Time'] * all_data['Episode_Sentiment']

# Group-based statistics
for cat in ['Genre', 'Publication_Day']:
    for col in ['Episode_Length_minutes', 'Host_Popularity_percentage', 'Guest_Popularity_percentage']:
        all_data[f'{cat}_{col}_mean'] = all_data.groupby(cat)[col].transform('mean')
        all_data[f'{cat}_{col}_std'] = all_data.groupby(cat)[col].transform('std')
        all_data[f'{cat}_{col}_min'] = all_data.groupby(cat)[col].transform('min')
        all_data[f'{cat}_{col}_max'] = all_data.groupby(cat)[col].transform('max')

In [15]:
# 8. Save encoded data to disk
encoded_path = os.path.join(data_dir, "encoded_data.csv")
all_data.to_csv(encoded_path, index=False)
all_data = pd.read_csv(encoded_path)

In [16]:
# 9. Split combined data back
X = all_data.iloc[:len(train)]
X_test = all_data.iloc[len(train):]

In [17]:

# 10. Save full training data and test features
X.to_csv(os.path.join(splits_dir, "X_full.csv"), index=False)
pd.DataFrame(y).to_csv(os.path.join(splits_dir, "y_full.csv"), index=False)
X_test.to_csv(os.path.join(splits_dir, "X_test.csv"), index=False)

In [18]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, module='ipykernel')


In [19]:
# Optuna objective for LightGBM tuning
def objective(trial):
    print(f"Trial #{trial.number} started")
    params = {
        'n_estimators': 200,
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.05, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 256),
        'max_depth': trial.suggest_int('max_depth', 5, 15),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 5.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 5.0),
        'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
        'random_state': 42,
        'n_jobs': -1,
        'verbosity': -1,
        'boosting_type': 'gbdt'
    }

    kf = KFold(n_splits=2, shuffle=True, random_state=42)
    val_scores = []
    for train_idx, valid_idx in kf.split(X):
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        model = lgb.LGBMRegressor(**params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_valid, y_valid)],
            eval_metric='rmse',
            callbacks=[
                lgb.early_stopping(30),
                LightGBMPruningCallback(trial, 'rmse'),
                lgb.log_evaluation(0)
            ]
        )
        pred = model.predict(X_valid)
        pred = np.clip(pred, 0, None)
        score = mean_squared_error(np.expm1(y_valid), np.expm1(pred), squared=False)
        val_scores.append(score)

    final_score = np.mean(val_scores)
    print(f"Trial #{trial.number} finished with score: {final_score}")
    return final_score

print("Starting Optuna tuning for LightGBM (max 10 minutes / 30 trials)...")
study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, timeout=600, n_trials=30)
print("Best trial:", study.best_trial.params)

best_params = study.best_trial.params
best_params.update({'n_estimators': 2000, 'random_state': 42, 'n_jobs': -1, 'verbosity': -1})

[I 2025-04-11 18:46:05,473] A new study created in memory with name: no-name-ea204d7c-e625-46e0-863d-d840110d3294


Starting Optuna tuning for LightGBM (max 10 minutes / 30 trials)...
Trial #0 started
Training until validation scores don't improve for 30 rounds
Did not meet early stopping. Best iteration is:
[200]	valid_0's rmse: 0.548915	valid_0's l2: 0.301307


/usr/local/lib/python3.10/dist-packages/optuna/trial/_trial.py:497: UserWarning: The reported value is ignored because this `step` 0 is already reported.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/optuna/trial/_trial.py:497: UserWarning: The reported value is ignored because this `step` 1 is already reported.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/optuna/trial/_trial.py:497: UserWarning: The reported value is ignored because this `step` 2 is already reported.
  warnings.warn(


Training until validation scores don't improve for 30 rounds


/usr/local/lib/python3.10/dist-packages/optuna/trial/_trial.py:497: UserWarning: The reported value is ignored because this `step` 3 is already reported.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/optuna/trial/_trial.py:497: UserWarning: The reported value is ignored because this `step` 4 is already reported.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/optuna/trial/_trial.py:497: UserWarning: The reported value is ignored because this `step` 5 is already reported.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/optuna/trial/_trial.py:497: UserWarning: The reported value is ignored because this `step` 6 is already reported.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/optuna/trial/_trial.py:497: UserWarning: The reported value is ignored because this `step` 7 is already reported.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/optuna/trial/_trial.py:497: UserWarning: The reported value is ignored because this `step` 8 is already r

...

Training until validation scores don't improve for 30 rounds


/usr/local/lib/python3.10/dist-packages/optuna/trial/_trial.py:497: UserWarning: The reported value is ignored because this `step` 3 is already reported.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/optuna/trial/_trial.py:497: UserWarning: The reported value is ignored because this `step` 4 is already reported.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/optuna/trial/_trial.py:497: UserWarning: The reported value is ignored because this `step` 5 is already reported.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/optuna/trial/_trial.py:497: UserWarning: The reported value is ignored because this `step` 6 is already reported.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/optuna/trial/_trial.py:497: UserWarning: The reported value is ignored because this `step` 7 is already reported.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/optuna/trial/_trial.py:497: UserWarning: The reported value is ignored because this `step` 8 is already r

Did not meet early stopping. Best iteration is:
[200]	valid_0's rmse: 0.393732	valid_0's l2: 0.155025


[I 2025-04-11 18:56:12,615] Trial 22 finished with value: 13.536571054496756 and parameters: {'learning_rate': 0.03275718967900387, 'num_leaves': 155, 'max_depth': 15, 'min_child_samples': 37, 'subsample': 0.8721001970842582, 'colsample_bytree': 0.9100599608637124, 'reg_alpha': 2.3112139630562947, 'reg_lambda': 1.0130942959942992, 'min_split_gain': 0.2760493943423752, 'bagging_freq': 4}. Best is trial 10 with value: 13.50893067052817.


Trial #22 finished with score: 13.536571054496756
Best trial: {'learning_rate': 0.04614599237890159, 'num_leaves': 203, 'max_depth': 11, 'min_child_samples': 39, 'subsample': 0.848068981182222, 'colsample_bytree': 0.963065918113007, 'reg_alpha': 4.798853729727099, 'reg_lambda': 1.4253994575600701, 'min_split_gain': 0.21246987241999954, 'bagging_freq': 7}


In [20]:
# 11. K-Fold Cross Validation Setup
kf = KFold(n_splits=5, shuffle=True, random_state=42)

fold = 1
val_scores = []
test_preds_log_lgb = np.zeros(len(X_test))
test_preds_log_cat = np.zeros(len(X_test))

In [21]:
for train_index, valid_index in kf.split(X):
    print(f"\nTraining fold {fold}...")
    X_train, X_valid = X.iloc[train_index], X.iloc[valid_index]
    y_train, y_valid = y.iloc[train_index], y.iloc[valid_index]

    # LightGBM with tuned parameters
    model_lgb = lgb.LGBMRegressor(**best_params)
    model_lgb.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], eval_metric='rmse')

    # CatBoost
    model_cat = CatBoostRegressor(
        iterations=2000,
        learning_rate=0.01,
        depth=10,
        l2_leaf_reg=3.0,
        loss_function='RMSE',
        verbose=False,
        random_state=42
    )
    model_cat.fit(X_train, y_train, eval_set=(X_valid, y_valid))

    # Predictions
    val_pred_lgb = model_lgb.predict(X_valid)
    val_pred_cat = model_cat.predict(X_valid)
    val_pred_avg = (val_pred_lgb + val_pred_cat) / 2

    val_pred_avg = pd.Series(val_pred_avg).replace([np.inf, -np.inf], np.nan).fillna(0)
    val_rmse = mean_squared_error(np.expm1(y_valid), np.expm1(val_pred_avg), squared=False)
    val_scores.append(val_rmse)
    print(f"Fold {fold} RMSE: {val_rmse:.5f}")

    # Test Predictions
    test_pred_lgb = model_lgb.predict(X_test)
    test_pred_cat = model_cat.predict(X_test)
    test_preds_log_lgb += pd.Series(test_pred_lgb).replace([np.inf, -np.inf], np.nan).fillna(0) / kf.n_splits
    test_preds_log_cat += pd.Series(test_pred_cat).replace([np.inf, -np.inf], np.nan).fillna(0) / kf.n_splits

    fold += 1


Training fold 1...
Fold 1 RMSE: 13.47158

Training fold 2...
Fold 2 RMSE: 13.53377

Training fold 3...
Fold 3 RMSE: 13.49804

Training fold 4...
Fold 4 RMSE: 13.47776

Training fold 5...
Fold 5 RMSE: 13.48876


In [22]:
# 12. Overall validation score
print(f"\nAverage CV RMSE: {np.mean(val_scores):.5f}")


Average CV RMSE: 13.49398


In [23]:
# 13. Create submission file
submission = sample_submission.copy()
final_preds_log = (test_preds_log_lgb + test_preds_log_cat) / 2
submission[target_column] = np.expm1(final_preds_log)
output_path = os.path.join(data_dir, "submission.csv")
submission.to_csv(output_path, index=False)
print(f"Submission file created at: {output_path}")

Submission file created at: /kaggle/working/submission.csv


In [27]:
submission.head(3)

,id,Listening_Time_minutes
0,750000,54.750906
1,750001,17.887299
2,750002,49.284748
